# Dia 02 - Data Transformation
objetivo: apresentar sobre as etapas de transformação dos dados armazenados no data warehouse

- paradigmas de fluxo de trabalho (etl x elt)
    - exemplificar as diferentes formas de transformação: em memória com python e dentro da engine com OLAP/DuckDB

- Limpeza e Qualidade de dados
    - data wrangling
    https://www.datacamp.com/pt/blog/what-is-data-wrangling
    - validações e testes de qualidade


- olap e agregações em sql
    - introduzir ao conceito de processamento analítico (OLAP)
    - apresentar a utilidade e funcionamento de CTEs do sql (GROUP BY, ORDER BY) e Windows Functions
    https://learnsql.com.br/blog/guia-de-funcoes-de-janela-do-sql/
    - apresentar a execução direta de sql sobre arquivos parquet sem necessidade de carga prévia dos dados

- modelagem dimensional e data marts
    - apresentar como tabelas de staging são convertidas em tabelas fato e tabelas dimensão (star schema/ kimball)
    - apresentar como ocorre o processo de criação de data marts para criação de subseções específicas da base de dados para atender a demandas específicas do negócio da instituição
    https://aws.amazon.com/pt/what-is/data-mart/

# DIA 2- Transformação de dados

Nesta etapa, vamos apresentar os principais processos envolvidos na transformação dos dados armazenados em um ambiente de Engenharia de Dados.

Ao longo do notebook, os dados de uma base fictícia de vendas passarão por processos de limpeza, validação, transformação e organização, até serem disponibilizados em estruturas adequadas para análises.

Serão utilizadas as bibliotecas Pandas e DuckDB para demonstrar diferentes abordagens de transformação, consultas analíticas em SQL e construção de uma modelagem dimensional.

## Preparando o ambiente

In [ ]:
#instalando as bibliotecas
%pip install pandas duckdb pyarrow

   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.1 MB 487.6 kB/s eta 0:00:34
   ---------------------------------------- 0.1/16.1 MB 930.9 kB/s eta 0:00:18
    --------------------------------------- 0.2/16.1 MB 1.4 MB/s eta 0:00:12
   - -------------------------------------- 0.5/16.1 MB 2.2 MB/s eta 0:00:07
   -- ------------------------------------- 0.9/16.1 MB 3.2 MB/s eta 0:00:05
   -- ------------------------------------- 1.1/16.1 MB 3.9 MB/s eta 0:00:04
   --- ------------------------------------ 1.6/16.1 MB 4.3 MB/s eta 0:00:04
   ----- ---------------------------------- 2.2/16.1 MB 5.6 MB/s eta 0:00:03
   ------ --------------------------------- 2.7/16.1 MB 6.2 MB/s eta 0:00:03
   -------- ------------------------------- 3.4/16.1 MB 6.9 MB/s eta 0:00:02
   ---------- ----------------------------- 4.1/16.1 MB 7.6 MB/s eta 0:00:02
   ------


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\ajb95\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
from pathlib import Path
import pandas as pd
import duckdb

caminho = Path("../../../data/modulo_03")
caminho.mkdir(parents=True, exist_ok=True)
con = duckdb.connect("../../../data/modulo_03/data_warehouse.duckdb")

## Criando a base de dados para o modulo

Antes de demonstrar os processos de transformação, precisamos de uma base de dados para trabalhar, por isso criamos uma pequena base fictícia de vendas, simulando dados provenientes de um sistema transacional.

**Observação:** os dados serão propositalmente criados com algumas inconsistências. Isso permitirá demonstrar, nas próximas etapas, como identificar e corrigir problemas de qualidade.

In [2]:
# Cria uma base fictícia de vendas.
# Os registros apresentam inconsistências que serão tratadas posteriormente.

dados_vendas = {
    "id_venda": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10],
    "data_venda": [
        "2026-01-10", "2026-02-11", "2026-03-12",
        "2026-01-13", "2026-02-14", "2026-04-15",
        "2026-01-16", "2026-03-17", "2026-04-18",
        "2026-02-19", "2026-03-19"
    ],
    "id_cliente": [101, 102, 103, 101, 104, 105, 102, 106, 107, 108, 108],
    "cliente": [
        "Ana Silva", " Carlos Souza ", "Mariana Lima",
        "Ana Silva", "João Santos", "Pedro Costa",
        "Carlos Souza", "Fernanda Alves", "Lucas Rocha",
        "Beatriz Mendes", "Beatriz Mendes"
    ],
    "produto": [
        "Notebook", "Mouse", "Teclado", "Monitor",
        "Notebook", "Mouse", "Teclado", "Monitor",
        "Notebook", "Mouse", "Mouse"
    ],
    "categoria": [
        "Eletrônicos", "Acessórios", "acessorios",
        "Eletrônicos", "Eletrônicos", "Acessórios",
        "Acessórios", "eletronicos", "Eletrônicos",
        "Acessórios", "Acessórios"
    ],
    "quantidade": [1, 2, 1, 1, 2, 3, 1, 1, 1, 2, 2],
    "preco_unitario": [3500, 50, 120, 1500, 3500, 50, 120, 1500, 3500, 50, 50],
    "desconto": [100, 0, 10, 50, 200, 0, 0, 50, 100, 5, 5]
}

# Converte o dicionário em um DataFrame.
df_bruto = pd.DataFrame(dados_vendas)

# Exibe os primeiros registros.
display(df_bruto)

# Exibe informações gerais sobre a base.
df_bruto.info()

,id_venda,data_venda,id_cliente,cliente,produto,categoria,quantidade,preco_unitario,desconto
0,1,2026-01-10,101,Ana Silva,Notebook,Eletrônicos,1,3500,100
1,2,2026-02-11,102,Carlos Souza,Mouse,Acessórios,2,50,0
2,3,2026-03-12,103,Mariana Lima,Teclado,acessorios,1,120,10
3,4,2026-01-13,101,Ana Silva,Monitor,Eletrônicos,1,1500,50
4,5,2026-02-14,104,João Santos,Notebook,Eletrônicos,2,3500,200
5,6,2026-04-15,105,Pedro Costa,Mouse,Acessórios,3,50,0
6,7,2026-01-16,102,Carlos Souza,Teclado,Acessórios,1,120,0
7,8,2026-03-17,106,Fernanda Alves,Monitor,eletronicos,1,1500,50
8,9,2026-04-18,107,Lucas Rocha,Notebook,Eletrônicos,1,3500,100
9,10,2026-02-19,108,Beatriz Mendes,Mouse,Acessórios,2,50,5


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_venda        11 non-null     int64 
 1   data_venda      11 non-null     object
 2   id_cliente      11 non-null     int64 
 3   cliente         11 non-null     object
 4   produto         11 non-null     object
 5   categoria       11 non-null     object
 6   quantidade      11 non-null     int64 
 7   preco_unitario  11 non-null     int64 
 8   desconto        11 non-null     int64 
dtypes: int64(5), object(4)
memory usage: 924.0+ bytes


In [3]:
# Salva os dados originais em um arquivo CSV.
# O arquivo representa uma fonte de dados que foi disponibilizada
# para o processo de Engenharia de Dados.

df_bruto.to_csv(
    "../../../data/modulo_03/vendas_brutas.csv",
    index=False,
    encoding="utf-8"
)

print("Arquivo de dados brutos armazenado com sucesso!")

Arquivo de dados brutos armazenado com sucesso!


# Paradigmas de fluxo de trabalho: ETL x ELT

## ETL — Transformação em memória
No modelo ETL, os dados são extraídos da origem, carregados na memória do computador e transformados utilizando ferramentas como Pandas. Somente depois da transformação são enviados ao destino.
Neste primeiro exemplo, utilizaremos o Pandas para realizar a limpeza e a transformação dos dados diretamente na memória do Python.

### Extração dos dados

In [4]:
# Lê o arquivo CSV que contém os dados brutos.
df_etl = pd.read_csv("../../../data/modulo_03/vendas_brutas.csv")

# Visualiza os dados extraídos.
display(df_etl.head())

print(f"Quantidade de registros extraídos: {len(df_etl)}")

,id_venda,data_venda,id_cliente,cliente,produto,categoria,quantidade,preco_unitario,desconto
0,1,2026-01-10,101,Ana Silva,Notebook,Eletrônicos,1,3500,100
1,2,2026-02-11,102,Carlos Souza,Mouse,Acessórios,2,50,0
2,3,2026-03-12,103,Mariana Lima,Teclado,acessorios,1,120,10
3,4,2026-01-13,101,Ana Silva,Monitor,Eletrônicos,1,1500,50
4,5,2026-02-14,104,João Santos,Notebook,Eletrônicos,2,3500,200


Quantidade de registros extraídos: 11


### Transformação dos dados

In [5]:
# Remove espaços desnecessários do inicio ou do final dos nomes dos clientes.
df_etl["cliente"] = df_etl["cliente"].str.strip()

# Padroniza os nomes das categorias, deixando todas com a primeira letra de cada palavra maiuscula e o resto minuscula.
df_etl["categoria"] = df_etl["categoria"].str.strip().str.title()

# Converte a coluna de data para o formato datetime.
df_etl["data_venda"] = pd.to_datetime(df_etl["data_venda"])

# Converte os campos numéricos para seus respectivos tipos.
df_etl["quantidade"] = pd.to_numeric(df_etl["quantidade"])
df_etl["preco_unitario"] = pd.to_numeric(df_etl["preco_unitario"])
df_etl["desconto"] = pd.to_numeric(df_etl["desconto"])

# Calcula o valor total de cada venda, considerando o desconto.
df_etl["valor_total"] = (
    df_etl["quantidade"] * df_etl["preco_unitario"]
) - df_etl["desconto"]

# Remove registros duplicados considerando o identificador da venda.
df_etl = df_etl.drop_duplicates(subset=["id_venda"], keep="first")

# Exibe os dados após a transformação.
display(df_etl)

,id_venda,data_venda,id_cliente,cliente,produto,categoria,quantidade,preco_unitario,desconto,valor_total
0,1,2026-01-10,101,Ana Silva,Notebook,Eletrônicos,1,3500,100,3400
1,2,2026-02-11,102,Carlos Souza,Mouse,Acessórios,2,50,0,100
2,3,2026-03-12,103,Mariana Lima,Teclado,Acessorios,1,120,10,110
3,4,2026-01-13,101,Ana Silva,Monitor,Eletrônicos,1,1500,50,1450
4,5,2026-02-14,104,João Santos,Notebook,Eletrônicos,2,3500,200,6800
5,6,2026-04-15,105,Pedro Costa,Mouse,Acessórios,3,50,0,150
6,7,2026-01-16,102,Carlos Souza,Teclado,Acessórios,1,120,0,120
7,8,2026-03-17,106,Fernanda Alves,Monitor,Eletronicos,1,1500,50,1450
8,9,2026-04-18,107,Lucas Rocha,Notebook,Eletrônicos,1,3500,100,3400
9,10,2026-02-19,108,Beatriz Mendes,Mouse,Acessórios,2,50,5,95


In [6]:
df_etl.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10 entries, 0 to 9
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_venda        10 non-null     int64         
 1   data_venda      10 non-null     datetime64[ns]
 2   id_cliente      10 non-null     int64         
 3   cliente         10 non-null     object        
 4   produto         10 non-null     object        
 5   categoria       10 non-null     object        
 6   quantidade      10 non-null     int64         
 7   preco_unitario  10 non-null     int64         
 8   desconto        10 non-null     int64         
 9   valor_total     10 non-null     int64         
dtypes: datetime64[ns](1), int64(6), object(3)
memory usage: 880.0+ bytes


### Load (Carregamento dos dados)

In [7]:
# Salva os dados transformados em um novo arquivo CSV.
# Esse arquivo representa o destino dos dados após o processo ETL.

df_etl.to_csv(
    "../../../data/modulo_03/vendas_etl.csv",
    index=False,
    encoding="utf-8"
)

print("Processo ETL concluído!")
print(f"Registros disponíveis após a transformação: {len(df_etl)}")

Processo ETL concluído!
Registros disponíveis após a transformação: 10


No exemplo, todas as transformações foram realizadas utilizando a memória do computador. O Pandas recebeu os dados, executou as operações de limpeza e cálculo e, posteriormente, salvou o resultado. Essa abordagem é adequada para conjuntos de dados menores e transformações que exigem manipulações complexas em Python. Entretanto, quando o volume de dados cresce significativamente, o processamento em memória pode se tornar um limitador.

## ELT — Transformação dentro da Engine
Diferentemente do ETL, no modelo ELT os dados brutos são carregados primeiro no ambiente de armazenamento. As transformações são executadas posteriormente, utilizando a capacidade de processamento do próprio mecanismo analítico.

Para demonstrar essa abordagem, utilizaremos o DuckDB, que permite executar comandos SQL diretamente sobre arquivos e tabelas, assim os dados brutos serão carregados no DuckDB sem passar previamente por um processo de limpeza no Pandas

### Load (Carregamento)


In [8]:
# Cria uma tabela para armazenar os dados brutos chamada vendas_raw.
# A função read_csv_auto permite que o DuckDB identifique
# automaticamente os tipos de dados do arquivo CSV.

con.execute("""
    CREATE OR REPLACE TABLE vendas_raw AS
    SELECT *
    FROM read_csv_auto('../../../data/modulo_03/vendas_brutas.csv')
""")

# Consulta os dados armazenados.
display(con.execute("""
    SELECT *
    FROM vendas_raw
""").df())

,id_venda,data_venda,id_cliente,cliente,produto,categoria,quantidade,preco_unitario,desconto
0,1,2026-01-10,101,Ana Silva,Notebook,Eletrônicos,1,3500,100
1,2,2026-02-11,102,Carlos Souza,Mouse,Acessórios,2,50,0
2,3,2026-03-12,103,Mariana Lima,Teclado,acessorios,1,120,10
3,4,2026-01-13,101,Ana Silva,Monitor,Eletrônicos,1,1500,50
4,5,2026-02-14,104,João Santos,Notebook,Eletrônicos,2,3500,200
5,6,2026-04-15,105,Pedro Costa,Mouse,Acessórios,3,50,0
6,7,2026-01-16,102,Carlos Souza,Teclado,Acessórios,1,120,0
7,8,2026-03-17,106,Fernanda Alves,Monitor,eletronicos,1,1500,50
8,9,2026-04-18,107,Lucas Rocha,Notebook,Eletrônicos,1,3500,100
9,10,2026-02-19,108,Beatriz Mendes,Mouse,Acessórios,2,50,5


### Transformação
As operações de limpeza e cálculo são realizadas diretamente pelo mecanismo de processamento do DuckDB.

In [9]:
# Cria uma tabela com os dados tratados.

con.execute("""
    CREATE OR REPLACE TABLE vendas_tratadas AS

    SELECT
        id_venda,
        CAST(data_venda AS DATE) AS data_venda,

        id_cliente,

        TRIM(cliente) AS cliente,

        TRIM(produto) AS produto,

        CASE
            WHEN LOWER(TRIM(categoria)) = 'acessorios'
                THEN 'Acessórios'
            WHEN LOWER(TRIM(categoria)) = 'eletronicos'
                THEN 'Eletrônicos'
            ELSE TRIM(categoria)
        END AS categoria,

        quantidade,
        preco_unitario,
        desconto,

        (quantidade * preco_unitario) - desconto AS valor_total

    FROM vendas_raw

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY id_venda
        ORDER BY id_venda
    ) = 1
""")

# Consulta o resultado da transformação.
display(con.execute("""
    SELECT *
    FROM vendas_tratadas
""").df())

,id_venda,data_venda,id_cliente,cliente,produto,categoria,quantidade,preco_unitario,desconto,valor_total
0,1,2026-01-10,101,Ana Silva,Notebook,Eletrônicos,1,3500,100,3400
1,2,2026-02-11,102,Carlos Souza,Mouse,Acessórios,2,50,0,100
2,3,2026-03-12,103,Mariana Lima,Teclado,Acessórios,1,120,10,110
3,4,2026-01-13,101,Ana Silva,Monitor,Eletrônicos,1,1500,50,1450
4,5,2026-02-14,104,João Santos,Notebook,Eletrônicos,2,3500,200,6800
5,6,2026-04-15,105,Pedro Costa,Mouse,Acessórios,3,50,0,150
6,7,2026-01-16,102,Carlos Souza,Teclado,Acessórios,1,120,0,120
7,8,2026-03-17,106,Fernanda Alves,Monitor,Eletrônicos,1,1500,50,1450
8,9,2026-04-18,107,Lucas Rocha,Notebook,Eletrônicos,1,3500,100,3400
9,10,2026-02-19,108,Beatriz Mendes,Mouse,Acessórios,2,50,5,95


Observe que o resultado é semelhante ao produzido no exemplo anterior com Pandas. Entretanto, neste caso, as operações foram executadas pelo DuckDB, utilizando comandos SQL.

Essa abordagem demonstra o princípio do ELT: os dados são carregados primeiro e transformados posteriormente dentro do mecanismo de processamento.

A comparação permite perceber que a escolha entre ETL e ELT depende das características do projeto, do volume de dados, das ferramentas disponíveis e da complexidade das transformações.

# Limpeza e qualidade de dados
Depois de compreender as diferentes abordagens de transformação, vamos aprofundar os processos de limpeza e validação dos dados.

Nesta etapa, trabalharemos com dois conceitos:
- Data Wrangling: preparação, limpeza, organização e enriquecimento dos dados.
- Validações e testes de qualidade: aplicação de regras para verificar se os dados atendem aos requisitos estabelecidos.

## Data Wrangling:
Vamos transformar dados brutos em informações organizadas e adequadas para análise, identificando inconsistências e aplicando operações de limpeza.

### Identificação de problemas

In [10]:
# Carrega novamente os dados brutos.
df_wrangling = pd.read_csv("../../../data/modulo_03/vendas_brutas.csv")

# Identifica a quantidade de valores ausentes por coluna.
print("Valores ausentes:")
display(df_wrangling.isnull().sum())

# Identifica a quantidade de registros duplicados.
print("Registros duplicados:")
print(df_wrangling.duplicated(subset=["id_venda"]).sum())

# Visualiza os valores únicos da coluna categoria.
print("Categorias encontradas:")
print(df_wrangling["categoria"].unique())

# Visualiza os nomes dos clientes.
print("Nomes dos clientes:")
display(df_wrangling["cliente"])

Valores ausentes:


id_venda          0
data_venda        0
id_cliente        0
cliente           0
produto           0
categoria         0
quantidade        0
preco_unitario    0
desconto          0
dtype: int64

Registros duplicados:
1
Categorias encontradas:
['Eletrônicos' 'Acessórios' 'acessorios' 'eletronicos']
Nomes dos clientes:


0          Ana Silva
1      Carlos Souza 
2       Mariana Lima
3          Ana Silva
4        João Santos
5        Pedro Costa
6       Carlos Souza
7     Fernanda Alves
8        Lucas Rocha
9     Beatriz Mendes
10    Beatriz Mendes
Name: cliente, dtype: object

### Tratamento dos dados

In [11]:
# Remove espaços em branco no início e no final dos textos.
df_wrangling["cliente"] = df_wrangling["cliente"].str.strip()
df_wrangling["produto"] = df_wrangling["produto"].str.strip()

# Padroniza as categorias, aqui, estamos tambem .
df_wrangling["categoria"] = (
    df_wrangling["categoria"]
    .str.strip()
    .replace({
        "acessorios": "Acessórios",
        "eletronicos": "Eletrônicos"
    })
)

# Remove registros duplicados utilizando o identificador da venda.
df_wrangling = df_wrangling.drop_duplicates(
    subset=["id_venda"],
    keep="first"
)

# Converte a coluna de data.
df_wrangling["data_venda"] = pd.to_datetime(
    df_wrangling["data_venda"],
    errors="coerce"
)

# Exibe os dados tratados.
display(df_wrangling)

,id_venda,data_venda,id_cliente,cliente,produto,categoria,quantidade,preco_unitario,desconto
0,1,2026-01-10,101,Ana Silva,Notebook,Eletrônicos,1,3500,100
1,2,2026-02-11,102,Carlos Souza,Mouse,Acessórios,2,50,0
2,3,2026-03-12,103,Mariana Lima,Teclado,Acessórios,1,120,10
3,4,2026-01-13,101,Ana Silva,Monitor,Eletrônicos,1,1500,50
4,5,2026-02-14,104,João Santos,Notebook,Eletrônicos,2,3500,200
5,6,2026-04-15,105,Pedro Costa,Mouse,Acessórios,3,50,0
6,7,2026-01-16,102,Carlos Souza,Teclado,Acessórios,1,120,0
7,8,2026-03-17,106,Fernanda Alves,Monitor,Eletrônicos,1,1500,50
8,9,2026-04-18,107,Lucas Rocha,Notebook,Eletrônicos,1,3500,100
9,10,2026-02-19,108,Beatriz Mendes,Mouse,Acessórios,2,50,5


### Enriquecimento dos dados

In [12]:
# Cria novas informações a partir dos dados existentes.

# Calcula o valor total da venda.
df_wrangling["valor_total"] = (
    df_wrangling["quantidade"] * df_wrangling["preco_unitario"]
) - df_wrangling["desconto"]

# Extrai o ano e o mês da data da venda.
df_wrangling["ano"] = df_wrangling["data_venda"].dt.year
df_wrangling["mes"] = df_wrangling["data_venda"].dt.month

# Cria uma classificação para o valor das vendas.
df_wrangling["faixa_venda"] = pd.cut(
    df_wrangling["valor_total"],
    bins=[-1, 500, 2000, float("inf")],
    labels=["Baixa", "Média", "Alta"]
)

# Exibe os dados enriquecidos.
display(df_wrangling)

,id_venda,data_venda,id_cliente,cliente,produto,categoria,quantidade,preco_unitario,desconto,valor_total,ano,mes,faixa_venda
0,1,2026-01-10,101,Ana Silva,Notebook,Eletrônicos,1,3500,100,3400,2026,1,Alta
1,2,2026-02-11,102,Carlos Souza,Mouse,Acessórios,2,50,0,100,2026,2,Baixa
2,3,2026-03-12,103,Mariana Lima,Teclado,Acessórios,1,120,10,110,2026,3,Baixa
3,4,2026-01-13,101,Ana Silva,Monitor,Eletrônicos,1,1500,50,1450,2026,1,Média
4,5,2026-02-14,104,João Santos,Notebook,Eletrônicos,2,3500,200,6800,2026,2,Alta
5,6,2026-04-15,105,Pedro Costa,Mouse,Acessórios,3,50,0,150,2026,4,Baixa
6,7,2026-01-16,102,Carlos Souza,Teclado,Acessórios,1,120,0,120,2026,1,Baixa
7,8,2026-03-17,106,Fernanda Alves,Monitor,Eletrônicos,1,1500,50,1450,2026,3,Média
8,9,2026-04-18,107,Lucas Rocha,Notebook,Eletrônicos,1,3500,100,3400,2026,4,Alta
9,10,2026-02-19,108,Beatriz Mendes,Mouse,Acessórios,2,50,5,95,2026,2,Baixa


O processo de Data Wrangling permitiu padronizar os textos, remover duplicidades, converter tipos de dados e criar novas informações a partir dos registros existentes, ou seja, faz parte da etapa de transforma"ao do ELT.

Essas operações tornam os dados mais organizados e facilitam sua utilização em análises posteriores, porém, não garantem por si só que eles estejam adequados para utilização, é necessário estabelecer regras que permitam verificar a integridade e a confiabilidade das informações, obedecendo o "Contrato de dados"

## Validações e testes de qualidade

Vamos criar algumas regras de validação para verificar se os registros atendem aos requisitos mínimos de qualidade.

### Criação de regras de validação

In [13]:
# Define regras de qualidade para a base de vendas.

def validar_dados(df):

    resultados = {}

    # Verifica se existem identificadores de venda duplicados.
    resultados["IDs de venda únicos"] = (
        not df["id_venda"].duplicated().any()
    )

    # Verifica se existem valores ausentes nas colunas obrigatórias.
    colunas_obrigatorias = [
        "id_venda",
        "data_venda",
        "id_cliente",
        "cliente",
        "produto",
        "quantidade",
        "preco_unitario"
    ]

    resultados["Campos obrigatórios preenchidos"] = (
        not df[colunas_obrigatorias].isnull().any().any()
    )

    # Verifica se as quantidades são positivas.
    resultados["Quantidades positivas"] = (
        (df["quantidade"] > 0).all()
    )

    # Verifica se os preços são positivos.
    resultados["Preços positivos"] = (
        (df["preco_unitario"] > 0).all()
    )

    # Verifica se as datas foram convertidas corretamente.
    resultados["Datas válidas"] = (
        df["data_venda"].notnull().all()
    )

    return resultados


# Executa as validações.
resultado_validacao = validar_dados(df_wrangling)

# Exibe o resultado de cada regra.
for regra, resultado in resultado_validacao.items():

    status = "APROVADO" if resultado else "REPROVADO"

    print(f"{regra}: {status}")

IDs de venda únicos: APROVADO
Campos obrigatórios preenchidos: APROVADO
Quantidades positivas: APROVADO
Preços positivos: APROVADO
Datas válidas: APROVADO


### Verificação geral de qualidade

In [14]:
# Verifica se todas as regras foram aprovadas.
qualidade_aprovada = all(resultado_validacao.values())

if qualidade_aprovada:

    print("Todos os testes foram aprovados.")
    print("Os dados estão aptos para a próxima etapa.")

else:

    print("Foram identificados problemas de qualidade.")
    print("Os dados precisam ser corrigidos antes de serem disponibilizados.")

Todos os testes foram aprovados.
Os dados estão aptos para a próxima etapa.


Essas validações permitem identificar problemas que poderiam comprometer as análises posteriores.

Em um pipeline de Engenharia de Dados, essas verificações podem ser automatizadas. Caso uma regra importante seja violada, a execução pode ser interrompida, impedindo que dados inconsistentes avancem para as próximas camadas do ambiente analítico.

# OLAP e agregações em SQL
Nessa seção, vamos trabalhar com o processamento analítico de dados (OLAP) e demonstrar como o SQL pode ser utilizado para produzir informações consolidadas a partir de grandes conjuntos de registros, utilizando o DuckDB para executar consultas SQL sobre a tabela vendas_tratadas, criada anteriormente.

Diferentemente dos sistemas OLTP, voltados às operações transacionais do cotidiano, os sistemas OLAP são projetados para consultas analíticas, agregações e exploração de dados.

## Agregações e ordenações: GROUP BY e ORDER BY
O comando GROUP BY permite agrupar registros que possuem características em comum, possibilitando calcular métricas consolidadas. Já o ORDER BY permite organizar os resultados de acordo com determinados critérios.

In [15]:
# Calcula o faturamento total de cada categoria de produtos.

consulta = """
    SELECT
        categoria,
        SUM(valor_total) AS faturamento_total,
        COUNT(*) AS quantidade_vendas

    FROM vendas_tratadas

    GROUP BY categoria

    ORDER BY faturamento_total DESC
"""

resultado = con.execute(consulta).df()

display(resultado)

,categoria,faturamento_total,quantidade_vendas
0,Eletrônicos,16500.0,5
1,Acessórios,575.0,5


In [16]:
# Agrupa as vendas por produto e calcula:
# - Quantidade total vendida.
# - Faturamento total.
# - Valor médio das vendas.

consulta = """
    SELECT
        produto,
        SUM(quantidade) AS quantidade_vendida,
        SUM(valor_total) AS faturamento,
        AVG(valor_total) AS ticket_medio

    FROM vendas_tratadas

    GROUP BY produto

    ORDER BY faturamento DESC
"""

resultado = con.execute(consulta).df()

display(resultado)

,produto,quantidade_vendida,faturamento,ticket_medio
0,Notebook,4.0,13600.0,4533.333333
1,Monitor,2.0,2900.0,1450.000000
2,Mouse,7.0,345.0,115.000000
3,Teclado,2.0,230.0,115.000000


## CTEs (Common Table Expressions)

As CTEs permitem organizar consultas SQL complexas em etapas lógicas, utilizando a cláusula WITH, ou seja, em vez de escrever uma única consulta extensa com várias subconsultas, podemos dividir o processamento em blocos menores, facilitando a leitura e a manutenção do código.

A consulta foi dividida em duas partes:
- faturamento_categoria: a CTE agrupa as vendas por categoria e calcula o faturamento.
- Consulta principal: utiliza o resultado da CTE para filtrar as categorias com faturamento superior a 1.000.

A CTE existe apenas durante a execução da consulta, então não é necessário criar uma tabela física no banco de dados.

In [17]:
# Utiliza uma CTE para calcular o faturamento de cada categoria
# e, posteriormente, identificar aquelas com faturamento superior
# a um determinado valor.

consulta = """
    WITH faturamento_categoria AS (

        SELECT
            categoria,
            SUM(valor_total) AS faturamento

        FROM vendas_tratadas

        GROUP BY categoria

    )

    SELECT
        categoria,
        faturamento

    FROM faturamento_categoria

    WHERE faturamento > 1000

    ORDER BY faturamento DESC
"""

resultado = con.execute(consulta).df()

display(resultado)

,categoria,faturamento
0,Eletrônicos,16500.0


## Funções de janela (Window Functions)

Permitem realizar cálculos sobre conjuntos de registros relacionados, mantendo as linhas individuais no resultado, preservando o detalhamento original, diferentemente do GROUP BY, que consolida vários registros em uma única linha por grupo.

Para demonstrar esse recurso, utilizaremos três funções: ROW_NUMBER( ), SUM( ) OVER e LAG( )

In [18]:
# Cria um ranking das vendas de cada categoria.
# O ROW_NUMBER atribui uma posição para cada registro,
# considerando o valor da venda.

consulta = """
    SELECT
        id_venda,
        produto,
        categoria,
        valor_total,

        ROW_NUMBER() OVER (
            PARTITION BY categoria
            ORDER BY valor_total DESC
        ) AS ranking_categoria

    FROM vendas_tratadas

    ORDER BY categoria, ranking_categoria
"""

resultado = con.execute(consulta).df()

display(resultado)

# PARTITION BY categoria: separa os registros por categoria.
# ORDER BY valor_total DESC: organiza as vendas da maior para a menor.
# ROW_NUMBER(): atribui uma posição para cada venda dentro de sua categoria.

,id_venda,produto,categoria,valor_total,ranking_categoria
0,6,Mouse,Acessórios,150,1
1,7,Teclado,Acessórios,120,2
2,3,Teclado,Acessórios,110,3
3,2,Mouse,Acessórios,100,4
4,10,Mouse,Acessórios,95,5
5,5,Notebook,Eletrônicos,6800,1
6,1,Notebook,Eletrônicos,3400,2
7,9,Notebook,Eletrônicos,3400,3
8,4,Monitor,Eletrônicos,1450,4
9,8,Monitor,Eletrônicos,1450,5


### Total acumulado de vendas

In [19]:
# Calcula o faturamento acumulado ao longo do período.
# A função SUM() OVER permite somar os valores progressivamente,
# sem eliminar os registros individuais.

consulta = """
    SELECT
        id_venda,
        data_venda,
        valor_total,

        SUM(valor_total) OVER (
            ORDER BY data_venda, id_venda
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS faturamento_acumulado

    FROM vendas_tratadas

    ORDER BY data_venda, id_venda
"""

resultado = con.execute(consulta).df()

display(resultado)

,id_venda,data_venda,valor_total,faturamento_acumulado
0,1,2026-01-10,3400,3400.0
1,4,2026-01-13,1450,4850.0
2,7,2026-01-16,120,4970.0
3,2,2026-02-11,100,5070.0
4,5,2026-02-14,6800,11870.0
5,10,2026-02-19,95,11965.0
6,3,2026-03-12,110,12075.0
7,8,2026-03-17,1450,13525.0
8,6,2026-04-15,150,13675.0
9,9,2026-04-18,3400,17075.0


### Comparação com a venda anterior

In [20]:
# Utiliza a função LAG() para recuperar o valor da venda anterior.
# Isso permite comparar os registros e identificar variações.

consulta = """
    SELECT
        id_venda,
        data_venda,
        valor_total,

        LAG(valor_total) OVER (
            ORDER BY data_venda, id_venda
        ) AS venda_anterior,

        valor_total - LAG(valor_total) OVER (
            ORDER BY data_venda, id_venda
        ) AS diferenca

    FROM vendas_tratadas

    ORDER BY data_venda, id_venda
"""

resultado = con.execute(consulta).df()

display(resultado)

,id_venda,data_venda,valor_total,venda_anterior,diferenca
0,1,2026-01-10,3400,<NA>,<NA>
1,4,2026-01-13,1450,3400,-1950
2,7,2026-01-16,120,1450,-1330
3,2,2026-02-11,100,120,-20
4,5,2026-02-14,6800,100,6700
5,10,2026-02-19,95,6800,-6705
6,3,2026-03-12,110,95,15
7,8,2026-03-17,1450,110,1340
8,6,2026-04-15,150,1450,-1300
9,9,2026-04-18,3400,150,3250


## Execução direta de SQL sobre arquivos Parquet

Normalmente, a execução de consultas SQL exige a existência de tabelas previamente armazenadas em um banco de dados Entretanto, mecanismos analíticos como o DuckDB permitem consultar arquivos Parquet diretamente, sem a necessidade de realizar uma carga prévia em tabelas.

### Exportação para parquet

In [21]:
# Exporta os dados tratados para o formato Parquet.
# O arquivo será utilizado diretamente nas consultas seguintes.

con.execute("""
    COPY vendas_tratadas
    TO '../../../data/modulo_03/vendas_tratadas.parquet'
    (FORMAT PARQUET)
""")

print("Arquivo Parquet criado com sucesso!")

Arquivo Parquet criado com sucesso!


### Consulta direta ao arquivo parquet

In [22]:
# Executa uma consulta SQL diretamente sobre o arquivo Parquet.
# Não é necessário criar uma tabela previamente no banco de dados.

consulta = """
    SELECT
        categoria,
        SUM(valor_total) AS faturamento_total,
        COUNT(*) AS quantidade_vendas

    FROM read_parquet('../../../data/modulo_03/vendas_tratadas.parquet')

    GROUP BY categoria

    ORDER BY faturamento_total DESC
"""

resultado = con.execute(consulta).df()

display(resultado)

,categoria,faturamento_total,quantidade_vendas
0,Eletrônicos,16500.0,5
1,Acessórios,575.0,5


### Filtro direto sobre o parquet

In [23]:
# Consulta somente as vendas com valor superior a 1.000.
# O DuckDB lê o arquivo e executa o filtro diretamente sobre ele.

consulta = """
    SELECT
        id_venda,
        data_venda,
        cliente,
        produto,
        valor_total

    FROM read_parquet('../../../data/modulo_03/vendas_tratadas.parquet')

    WHERE valor_total > 1000

    ORDER BY valor_total DESC
"""

resultado = con.execute(consulta).df()

display(resultado)

,id_venda,data_venda,cliente,produto,valor_total
0,5,2026-02-14,João Santos,Notebook,6800
1,1,2026-01-10,Ana Silva,Notebook,3400
2,9,2026-04-18,Lucas Rocha,Notebook,3400
3,4,2026-01-13,Ana Silva,Monitor,1450
4,8,2026-03-17,Fernanda Alves,Monitor,1450


Observe que foi possível executar operações de agrupamento e filtragem diretamente sobre o arquivo Parquet.

Esse recurso é especialmente útil em arquiteturas modernas de Engenharia de Dados, nas quais os arquivos podem permanecer armazenados em um Data Lake, enquanto ferramentas analíticas realizam consultas diretamente sobre eles.

# Modelagem dimensional e Data Marts

Depois de realizar a limpeza e as transformações, os dados precisam ser organizados para facilitar seu consumo pelas ferramentas de Business Intelligence (BI).

Nesta etapa, vamos transformar a tabela de vendas tratadas em uma estrutura dimensional, utilizando o modelo Star Schema.

## Construção do Star Schema

A modelagem dimensional organiza os dados em duas categorias principais:
- Tabelas Fato: armazenam os eventos do negócio e suas métricas, como quantidade vendida, desconto e valor total.
- Tabelas Dimensão: armazenam informações descritivas que permitem contextualizar os eventos, como clientes, produtos e datas.

### Criação da dimensão de clientes

In [24]:
# Cada cliente aparece uma única vez na tabela.
# O identificador do cliente funciona como chave primária.

con.execute("""
    CREATE OR REPLACE TABLE dim_cliente AS

    SELECT DISTINCT
        id_cliente,
        cliente AS nome_cliente

    FROM vendas_tratadas
""")

display(con.execute("""
    SELECT *
    FROM dim_cliente
""").df())

,id_cliente,nome_cliente
0,106,Fernanda Alves
1,105,Pedro Costa
2,101,Ana Silva
3,103,Mariana Lima
4,104,João Santos
5,107,Lucas Rocha
6,102,Carlos Souza
7,108,Beatriz Mendes


### Criação da dimensão de produtos

In [25]:
# Cria a dimensão de produtos.
# Cada produto recebe um identificador único (id_produto),
# que será utilizado como chave para relacionar a dimensão
# com a tabela fato.

con.execute("""
    CREATE OR REPLACE TABLE dim_produto AS

    SELECT
        ROW_NUMBER() OVER (
            ORDER BY produto, categoria
        ) AS id_produto,

        produto AS nome_produto,
        categoria

    FROM (
        SELECT DISTINCT
            produto,
            categoria
        FROM vendas_tratadas
    )
""")

display(con.execute("""
    SELECT *
    FROM dim_produto
""").df())

,id_produto,nome_produto,categoria
0,1,Monitor,Eletrônicos
1,2,Mouse,Acessórios
2,3,Notebook,Eletrônicos
3,4,Teclado,Acessórios


### Criação da dimensão de datas

In [26]:
# Cria uma dimensão de datas.
# Essa tabela permite realizar análises temporais,
# agrupando as vendas por dia, mês ou ano.

con.execute("""
    CREATE OR REPLACE TABLE dim_data AS

    SELECT DISTINCT
        CAST(
            STRFTIME(data_venda, '%Y%m%d') AS INTEGER
        ) AS id_data,

        data_venda,

        EXTRACT(DAY FROM data_venda) AS dia,
        EXTRACT(MONTH FROM data_venda) AS mes,
        EXTRACT(YEAR FROM data_venda) AS ano

    FROM vendas_tratadas
""")

display(con.execute("""
    SELECT *
    FROM dim_data
    ORDER BY data_venda
""").df())

,id_data,data_venda,dia,mes,ano
0,20260110,2026-01-10,10,1,2026
1,20260113,2026-01-13,13,1,2026
2,20260116,2026-01-16,16,1,2026
3,20260211,2026-02-11,11,2,2026
4,20260214,2026-02-14,14,2,2026
5,20260219,2026-02-19,19,2,2026
6,20260312,2026-03-12,12,3,2026
7,20260317,2026-03-17,17,3,2026
8,20260415,2026-04-15,15,4,2026
9,20260418,2026-04-18,18,4,2026


### Criação da tabela fato

In [27]:
# Cria a tabela fato de vendas.
# A tabela armazena os eventos de vendas e suas métricas,
# utilizando as chaves estrangeiras das dimensões.

con.execute("""
    CREATE OR REPLACE TABLE fato_vendas AS

    SELECT
        v.id_venda,

        v.id_cliente,

        p.id_produto,

        d.id_data,

        v.quantidade,
        v.preco_unitario,
        v.desconto,
        v.valor_total

    FROM vendas_tratadas v

    INNER JOIN dim_produto p
        ON v.produto = p.nome_produto
        AND v.categoria = p.categoria

    INNER JOIN dim_cliente c
        ON v.id_cliente = c.id_cliente

    INNER JOIN dim_data d
        ON CAST(
            STRFTIME(v.data_venda, '%Y%m%d') AS INTEGER
        ) = d.id_data
""")

display(con.execute("""
    SELECT *
    FROM fato_vendas
""").df())

,id_venda,id_cliente,id_produto,id_data,quantidade,preco_unitario,desconto,valor_total
0,1,101,3,20260110,1,3500,100,3400
1,2,102,2,20260211,2,50,0,100
2,3,103,4,20260312,1,120,10,110
3,4,101,1,20260113,1,1500,50,1450
4,5,104,3,20260214,2,3500,200,6800
5,6,105,2,20260415,3,50,0,150
6,7,102,4,20260116,1,120,0,120
7,8,106,1,20260317,1,1500,50,1450
8,9,107,3,20260418,1,3500,100,3400
9,10,108,2,20260219,2,50,5,95


Agora temos uma tabela Fato conectada às três tabelas Dimensão:
- fato_vendas: registra os eventos de vendas e suas métricas.
- dim_cliente: apresenta as informações dos clientes.
- dim_produto: apresenta os produtos e suas categorias.
- dim_data: apresenta os atributos temporais.

Essa organização caracteriza um Star Schema, pois a tabela Fato fica no centro da estrutura, conectada às dimensões que fornecem o contexto para as análises.

### Consulta utilizando as dimensões

In [28]:
# Realiza uma consulta relacionando a tabela Fato
# com as dimensões de clientes, produtos e datas.

consulta = """
    SELECT
        c.nome_cliente,
        p.nome_produto,
        p.categoria,
        d.ano,
        d.mes,

        SUM(f.quantidade) AS quantidade_vendida,
        SUM(f.valor_total) AS faturamento

    FROM fato_vendas f

    INNER JOIN dim_cliente c
        ON f.id_cliente = c.id_cliente

    INNER JOIN dim_produto p
        ON f.id_produto = p.id_produto

    INNER JOIN dim_data d
        ON f.id_data = d.id_data

    GROUP BY
        c.nome_cliente,
        p.nome_produto,
        p.categoria,
        d.ano,
        d.mes

    ORDER BY faturamento DESC
"""

resultado = con.execute(consulta).df()

display(resultado)

,nome_cliente,nome_produto,categoria,ano,mes,quantidade_vendida,faturamento
0,João Santos,Notebook,Eletrônicos,2026,2,2.0,6800.0
1,Lucas Rocha,Notebook,Eletrônicos,2026,4,1.0,3400.0
2,Ana Silva,Notebook,Eletrônicos,2026,1,1.0,3400.0
3,Fernanda Alves,Monitor,Eletrônicos,2026,3,1.0,1450.0
4,Ana Silva,Monitor,Eletrônicos,2026,1,1.0,1450.0
5,Pedro Costa,Mouse,Acessórios,2026,4,3.0,150.0
6,Carlos Souza,Teclado,Acessórios,2026,1,1.0,120.0
7,Mariana Lima,Teclado,Acessórios,2026,3,1.0,110.0
8,Carlos Souza,Mouse,Acessórios,2026,2,2.0,100.0
9,Beatriz Mendes,Mouse,Acessórios,2026,2,2.0,95.0


## Criação de Data Marts

Quando um Data Warehouse cresce, diferentes departamentos podem precisar acessar conjuntos específicos de informações. Os Data Marts permitem criar subconjuntos organizados de dados, direcionados às necessidades de cada área da instituição

### Data Mart Comercial

Precisa acompanhar o faturamento e o desempenho dos produtos

In [29]:
# Cria um Data Mart direcionado ao departamento comercial.
# O objetivo é disponibilizar informações sobre produtos,
# categorias e faturamento.

con.execute("""
    CREATE OR REPLACE TABLE mart_comercial AS

    SELECT
        p.nome_produto,
        p.categoria,

        d.ano,
        d.mes,

        SUM(f.quantidade) AS quantidade_vendida,
        SUM(f.valor_total) AS faturamento_total

    FROM fato_vendas f

    INNER JOIN dim_produto p
        ON f.id_produto = p.id_produto

    INNER JOIN dim_data d
        ON f.id_data = d.id_data

    GROUP BY
        p.nome_produto,
        p.categoria,
        d.ano,
        d.mes
""")

# Consulta o Data Mart Comercial.
display(con.execute("""
    SELECT *
    FROM mart_comercial
    ORDER BY faturamento_total DESC
""").df())

,nome_produto,categoria,ano,mes,quantidade_vendida,faturamento_total
0,Notebook,Eletrônicos,2026,2,2.0,6800.0
1,Notebook,Eletrônicos,2026,4,1.0,3400.0
2,Notebook,Eletrônicos,2026,1,1.0,3400.0
3,Monitor,Eletrônicos,2026,3,1.0,1450.0
4,Monitor,Eletrônicos,2026,1,1.0,1450.0
5,Mouse,Acessórios,2026,2,4.0,195.0
6,Mouse,Acessórios,2026,4,3.0,150.0
7,Teclado,Acessórios,2026,1,1.0,120.0
8,Teclado,Acessórios,2026,3,1.0,110.0


### Data Mart de Gestão

Precisa acompanhar o desempenho das vendas e dos clientes.

In [30]:
# Cria um Data Mart direcionado ao departamento de gestão.
# O objetivo é disponibilizar indicadores sobre os clientes
# e o desempenho das vendas.

con.execute("""
    CREATE OR REPLACE TABLE mart_gestao AS

    SELECT
        c.id_cliente,
        c.nome_cliente,

        COUNT(f.id_venda) AS total_vendas,

        SUM(f.quantidade) AS quantidade_itens,

        SUM(f.valor_total) AS faturamento_total,

        AVG(f.valor_total) AS ticket_medio

    FROM fato_vendas f

    INNER JOIN dim_cliente c
        ON f.id_cliente = c.id_cliente

    GROUP BY
        c.id_cliente,
        c.nome_cliente
""")

# Consulta o Data Mart de Gestão.
display(con.execute("""
    SELECT *
    FROM mart_gestao
    ORDER BY faturamento_total DESC
""").df())

,id_cliente,nome_cliente,total_vendas,quantidade_itens,faturamento_total,ticket_medio
0,104,João Santos,1,2.0,6800.0,6800.0
1,101,Ana Silva,2,2.0,4850.0,2425.0
2,107,Lucas Rocha,1,1.0,3400.0,3400.0
3,106,Fernanda Alves,1,1.0,1450.0,1450.0
4,102,Carlos Souza,2,3.0,220.0,110.0
5,105,Pedro Costa,1,3.0,150.0,150.0
6,103,Mariana Lima,1,1.0,110.0,110.0
7,108,Beatriz Mendes,1,2.0,95.0,95.0


Note que os dois Data Marts foram construídos a partir da mesma estrutura dimensional, mas apresentam informações diferentes, pois atendem a necessidades específicas. Essa separação facilita o acesso às informações, organiza o ambiente analítico e permite estabelecer permissões específicas para cada departamento.

In [31]:
# Lista as tabelas disponíveis no banco de dados.
# Permite verificar as estruturas criadas durante o notebook.

tabelas = con.execute("""
    SHOW TABLES
""").df()

display(tabelas)

# Encerra a conexão com o DuckDB.
con.close()

print("Notebook finalizado. Conexão com o DuckDB encerrada.")

,name
0,dim_cliente
1,dim_data
2,dim_produto
3,fato_vendas
4,mart_comercial
5,mart_gestao
6,vendas_raw
7,vendas_tratadas


Notebook finalizado. Conexão com o DuckDB encerrada.
